# MindSync Stage 3: Multimodal CMAF Fusion + JSD Incongruence

**Goal:** Train Cross-Modal Attention Fusion (CMAF) that combines the trained text (RoBERTa-Large) and audio (wav2vec2-Large) embeddings into a single 4-cluster prediction, AND emits Jensen-Shannon Divergence (JSD) between the two streams as an incongruence score.

**Smart trick:** Pre-compute frozen text/audio embeddings ONCE, then train the small fusion module on cached embeddings. Reduces training from ~3-4 hours to ~30 min on T4.

**Pre-reqs:**
1. Stage 1 (text model) uploaded at `Ubaida1/mindsync-text-model` ✓
2. Stage 2 (audio model) uploaded at `Ubaida1/mindsync-audio-model` ✓

**Steps before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. 🔑 Secrets → Add **HF_TOKEN** (write access)
3. Runtime → Run all

## 1 · Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/mindsync')
ROOT.mkdir(exist_ok=True)
CKPT_DIR = ROOT / 'fusion_ckpt'
CKPT_DIR.mkdir(exist_ok=True)
EMB_DIR = ROOT / 'embeddings'
EMB_DIR.mkdir(exist_ok=True)
print('Checkpoints:', CKPT_DIR)
print('Embeddings cache:', EMB_DIR)

In [ ]:
!pip install -q transformers==4.45.2 datasets==2.21.0 librosa==0.10.2 soundfile scikit-learn huggingface_hub==0.25.2
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
from google.colab import userdata
import os
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN secret missing'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
print('HF token loaded ✓')

## 2 · Cluster mapping + RAVDESS download

In [ ]:
CLUSTER_NAMES = ['Distress', 'Resilience', 'Aggression', 'Ambiguity']
CLUSTER_TO_IDX = {c: i for i, c in enumerate(CLUSTER_NAMES)}

EMOTION_TO_CLUSTER = {
    'grief': 'Distress', 'nervousness': 'Distress', 'fear': 'Distress',
    'sadness': 'Distress', 'remorse': 'Distress',
    'joy': 'Resilience', 'admiration': 'Resilience', 'excitement': 'Resilience',
    'relief': 'Resilience', 'amusement': 'Resilience', 'approval': 'Resilience',
    'curiosity': 'Resilience', 'desire': 'Resilience', 'gratitude': 'Resilience',
    'love': 'Resilience', 'optimism': 'Resilience', 'pride': 'Resilience',
    'realization': 'Resilience',
    'anger': 'Aggression', 'annoyance': 'Aggression', 'disgust': 'Aggression',
    'disapproval': 'Aggression', 'embarrassment': 'Aggression',
    'confusion': 'Ambiguity', 'disappointment': 'Ambiguity', 'surprise': 'Ambiguity',
    'caring': 'Ambiguity', 'neutral': 'Ambiguity',
}

RAVDESS_EMOTION_TO_CLUSTER = {
    1: 3,  # neutral → Ambiguity
    2: 1,  # calm → Resilience
    3: 1,  # happy → Resilience
    4: 0,  # sad → Distress
    5: 2,  # angry → Aggression
    6: 0,  # fearful → Distress
    7: 2,  # disgust → Aggression
    8: 3,  # surprised → Ambiguity
}
print('Cluster mappings ready')

In [ ]:
import pathlib, urllib.request, zipfile
RAVDESS_URL = 'https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip'
DATA_DIR = ROOT / 'ravdess'
DATA_DIR.mkdir(exist_ok=True)
ZIP = DATA_DIR / 'ravdess.zip'
if not ZIP.exists() or ZIP.stat().st_size < 100_000_000:
    print('Downloading RAVDESS...')
    urllib.request.urlretrieve(RAVDESS_URL, ZIP)
AUDIO_ROOT = DATA_DIR / 'extracted'
if not AUDIO_ROOT.exists() or len(list(AUDIO_ROOT.rglob('*.wav'))) < 1000:
    AUDIO_ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP) as z: z.extractall(AUDIO_ROOT)
wavs = sorted(AUDIO_ROOT.rglob('*.wav'))
print(f'RAVDESS files: {len(wavs)}')

## 3 · Download pretrained Stage 1 + Stage 2 checkpoints

In [ ]:
from huggingface_hub import hf_hub_download
TEXT_CKPT = hf_hub_download('Ubaida1/mindsync-text-model', 'best_text_model.pt')
AUDIO_CKPT = hf_hub_download('Ubaida1/mindsync-audio-model', 'best_audio_model.pt')
print('Text ckpt:', TEXT_CKPT)
print('Audio ckpt:', AUDIO_CKPT)

## 4 · Define encoder architectures (matching saved checkpoints)

In [ ]:
import torch, torch.nn as nn
from transformers import RobertaModel, Wav2Vec2Model

# Match Stage 1 notebook exactly
class MindSyncTextModel(nn.Module):
    def __init__(self, model_name='roberta-large', num_classes=4, dropout=0.1):
        super().__init__()
        self.encoder = nn.Module()
        self.encoder.encoder = RobertaModel.from_pretrained(model_name)
        self.encoder.hidden_size = self.encoder.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Module()
        h = self.encoder.hidden_size
        self.classifier.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(h, h // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(h // 2, num_classes),
        )
    def forward(self, input_ids, attention_mask):
        out = self.encoder.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        logits = self.classifier.classifier(cls)
        return {'embedding': cls, 'logits': logits}

# Match Stage 2 notebook exactly
class Wav2VecAudioEncoder(nn.Module):
    def __init__(self, model_name='facebook/wav2vec2-large-960h', dropout=0.1):
        super().__init__()
        self.encoder = Wav2Vec2Model.from_pretrained(model_name)
        self.hidden_size = self.encoder.config.hidden_size
        if hasattr(self.encoder, 'freeze_feature_encoder'):
            self.encoder.freeze_feature_encoder()
        self.dropout = nn.Dropout(dropout)
    def forward(self, iv):
        return self.dropout(self.encoder(input_values=iv).last_hidden_state.mean(dim=1))

class AudioClassificationHead(nn.Module):
    def __init__(self, hidden_size=1024, num_classes=4, dropout=0.1):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden_size, hidden_size//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_size//2, num_classes),
        )
    def forward(self, e): return self.classifier(e)

class MindSyncAudioModel(nn.Module):
    def __init__(self, model_name='facebook/wav2vec2-large-960h', num_classes=4):
        super().__init__()
        self.encoder = Wav2VecAudioEncoder(model_name=model_name)
        self.classifier = AudioClassificationHead(self.encoder.hidden_size, num_classes)
    def forward(self, iv):
        emb = self.encoder(iv)
        return {'embedding': emb, 'logits': self.classifier(emb)}
print('Model classes defined')

In [ ]:
device = torch.device('cuda')

text_model = MindSyncTextModel().to(device)
text_sd = torch.load(TEXT_CKPT, map_location=device, weights_only=False)
if isinstance(text_sd, dict) and 'model_state_dict' in text_sd: text_sd = text_sd['model_state_dict']
text_model.load_state_dict(text_sd)
text_model.eval()
for p in text_model.parameters(): p.requires_grad = False
print('Text model loaded + frozen')

audio_model = MindSyncAudioModel().to(device)
audio_sd = torch.load(AUDIO_CKPT, map_location=device, weights_only=False)
if isinstance(audio_sd, dict) and 'model_state_dict' in audio_sd: audio_sd = audio_sd['model_state_dict']
audio_model.load_state_dict(audio_sd)
audio_model.eval()
for p in audio_model.parameters(): p.requires_grad = False
print('Audio model loaded + frozen')

## 5 · Pre-compute embeddings (cache to Drive)

Embeddings ONCE so fusion training is fast. Saved to Drive so re-runs skip this step.

In [ ]:
# === Pre-compute RAVDESS audio embeddings ===
import numpy as np, librosa, torch
AUDIO_EMB_PATH = EMB_DIR / 'ravdess_embeddings.pt'

if AUDIO_EMB_PATH.exists():
    cached = torch.load(AUDIO_EMB_PATH, weights_only=False)
    audio_embs, audio_clusters, audio_actors = cached['emb'], cached['cluster'], cached['actor']
    print(f'Cached: {len(audio_embs)} audio embeddings')
else:
    audio_embs, audio_clusters, audio_actors = [], [], []
    TARGET_SR, MAX_SAMPLES = 16_000, 80_000
    import time; t0 = time.time()
    for i, wav_path in enumerate(wavs):
        parts = wav_path.stem.split('-')
        if len(parts) != 7: continue
        emotion, actor = int(parts[2]), int(parts[6])
        cluster = RAVDESS_EMOTION_TO_CLUSTER[emotion]
        wav, _ = librosa.load(str(wav_path), sr=TARGET_SR, mono=True)
        wav, _ = librosa.effects.trim(wav, top_db=30)
        if len(wav) > MAX_SAMPLES:
            s = (len(wav) - MAX_SAMPLES) // 2; wav = wav[s:s+MAX_SAMPLES]
        else:
            wav = np.pad(wav, (0, MAX_SAMPLES - len(wav)))
        wav = wav / (np.abs(wav).max() + 1e-8)
        iv = torch.tensor(wav, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad(), torch.cuda.amp.autocast():
            emb = audio_model.encoder(iv).cpu().float().squeeze(0)
        audio_embs.append(emb)
        audio_clusters.append(cluster)
        audio_actors.append(actor)
        if (i+1) % 100 == 0:
            print(f'  {i+1}/{len(wavs)} | {time.time()-t0:.0f}s', flush=True)
    audio_embs = torch.stack(audio_embs)
    audio_clusters = torch.tensor(audio_clusters, dtype=torch.long)
    audio_actors = torch.tensor(audio_actors, dtype=torch.long)
    torch.save({'emb': audio_embs, 'cluster': audio_clusters, 'actor': audio_actors}, AUDIO_EMB_PATH)
    print(f'Saved {len(audio_embs)} audio embeddings to {AUDIO_EMB_PATH}')
print('Audio embedding dim:', audio_embs.shape)

In [ ]:
# === Pre-compute GoEmotions TEXT embeddings ===
from datasets import load_dataset
from transformers import AutoTokenizer
TEXT_EMB_PATH = EMB_DIR / 'goemotions_embeddings.pt'
MAX_PER_CLUSTER = 1500  # balance dataset; total ~6000 samples

if TEXT_EMB_PATH.exists():
    cached = torch.load(TEXT_EMB_PATH, weights_only=False)
    text_embs, text_clusters = cached['emb'], cached['cluster']
    print(f'Cached: {len(text_embs)} text embeddings')
else:
    ds = load_dataset('google-research-datasets/go_emotions', 'simplified')
    label_names = ds['train'].features['labels'].feature.names
    def to_cluster(label_ids):
        if not label_ids: return None
        return CLUSTER_TO_IDX[EMOTION_TO_CLUSTER.get(label_names[label_ids[0]], 'Ambiguity')]
    
    # Collect balanced samples
    from collections import defaultdict
    bucket = defaultdict(list)
    for split in ['train', 'validation']:
        for item in ds[split]:
            c = to_cluster(item['labels'])
            if c is None: continue
            if len(bucket[c]) < MAX_PER_CLUSTER:
                bucket[c].append(item['text'])
    
    texts, clusters = [], []
    for c, ts in bucket.items():
        for t in ts:
            texts.append(t); clusters.append(c)
    print(f'Balanced text samples: {len(texts)} ({dict({c: len(bucket[c]) for c in bucket})})')
    
    tok = AutoTokenizer.from_pretrained('roberta-large')
    text_embs = []
    BATCH = 16
    import time; t0 = time.time()
    for i in range(0, len(texts), BATCH):
        batch_texts = texts[i:i+BATCH]
        enc = tok(batch_texts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
        with torch.no_grad(), torch.cuda.amp.autocast():
            out = text_model.encoder.encoder(input_ids=enc['input_ids'].to(device),
                                              attention_mask=enc['attention_mask'].to(device))
            embs = out.last_hidden_state[:, 0, :].cpu().float()
        text_embs.append(embs)
        if (i // BATCH + 1) % 50 == 0:
            print(f'  {i+BATCH}/{len(texts)} | {time.time()-t0:.0f}s', flush=True)
    text_embs = torch.cat(text_embs)
    text_clusters = torch.tensor(clusters, dtype=torch.long)
    torch.save({'emb': text_embs, 'cluster': text_clusters}, TEXT_EMB_PATH)
    print(f'Saved {len(text_embs)} text embeddings to {TEXT_EMB_PATH}')
print('Text embedding dim:', text_embs.shape)

## 6 · Build paired multimodal dataset

For each cluster, pair text embeddings with audio embeddings of the same cluster.

In [ ]:
import random
random.seed(42)
torch.manual_seed(42)

# Speaker-independent split for audio: actors 1-18 train, 19-21 val, 22-24 test
audio_train_mask = audio_actors <= 18
audio_val_mask   = (audio_actors >= 19) & (audio_actors <= 21)
audio_test_mask  = audio_actors >= 22

# Text: split by index (80/10/10)
n_text = len(text_embs)
perm = torch.randperm(n_text)
n_tr = int(0.8 * n_text); n_vl = int(0.1 * n_text)
text_train_idx = perm[:n_tr]; text_val_idx = perm[n_tr:n_tr+n_vl]; text_test_idx = perm[n_tr+n_vl:]

def build_pairs(text_idx, audio_mask, n_pairs):
    """For each text sample, pair with a random audio of same cluster."""
    pairs = []
    audio_pool = {c: [i for i in audio_mask.nonzero().squeeze(-1).tolist() if audio_clusters[i].item() == c] for c in range(4)}
    # Filter clusters that have audio
    text_idx_filtered = [i.item() for i in text_idx if audio_pool[text_clusters[i].item()]]
    chosen = random.sample(text_idx_filtered, min(n_pairs, len(text_idx_filtered)))
    for ti in chosen:
        c = text_clusters[ti].item()
        ai = random.choice(audio_pool[c])
        pairs.append((ti, ai, c))
    return pairs

train_pairs = build_pairs(text_train_idx, audio_train_mask, n_pairs=4000)
val_pairs   = build_pairs(text_val_idx,   audio_val_mask,   n_pairs=600)
test_pairs  = build_pairs(text_test_idx,  audio_test_mask,  n_pairs=600)
print(f'Pairs — train: {len(train_pairs)} | val: {len(val_pairs)} | test: {len(test_pairs)}')

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np

class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        ti, ai, c = self.pairs[i]
        return {'text_emb': text_embs[ti], 'audio_emb': audio_embs[ai], 'labels': torch.tensor(c, dtype=torch.long)}

train_ds = PairDataset(train_pairs)
val_ds   = PairDataset(val_pairs)
test_ds  = PairDataset(test_pairs)

# Class-balanced sampler for train
labels_arr = np.array([p[2] for p in train_pairs])
cls_counts = np.bincount(labels_arr, minlength=4)
weights = 1.0 / cls_counts[labels_arr]
sampler = WeightedRandomSampler(weights, num_samples=len(labels_arr), replacement=True)

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False)
print('Loaders ready, batches/epoch:', len(train_loader))

## 7 · CMAF fusion module + JSD

In [ ]:
import torch.nn.functional as F

class CMAFFusion(nn.Module):
    """Cross-Modal Attention Fusion: bidirectional cross-attention between
    text and audio embeddings, then a classifier on the concatenation.
    Also exposes per-stream heads for JSD computation."""
    def __init__(self, text_dim=1024, audio_dim=1024, d=256, num_classes=4, dropout=0.1, num_heads=4):
        super().__init__()
        self.text_proj  = nn.Linear(text_dim,  d)
        self.audio_proj = nn.Linear(audio_dim, d)
        self.t2a = nn.MultiheadAttention(d, num_heads, dropout=dropout, batch_first=True)
        self.a2t = nn.MultiheadAttention(d, num_heads, dropout=dropout, batch_first=True)
        self.norm_t = nn.LayerNorm(d)
        self.norm_a = nn.LayerNorm(d)
        self.fuse = nn.Sequential(
            nn.Linear(2*d, d), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d, num_classes),
        )
        # Auxiliary per-stream heads (for JSD at inference)
        self.head_text  = nn.Linear(d, num_classes)
        self.head_audio = nn.Linear(d, num_classes)

    def forward(self, e_text, e_audio):
        t = self.text_proj(e_text).unsqueeze(1)
        a = self.audio_proj(e_audio).unsqueeze(1)
        t_attn, _ = self.t2a(t, a, a)
        a_attn, _ = self.a2t(a, t, t)
        t = self.norm_t(t + t_attn).squeeze(1)
        a = self.norm_a(a + a_attn).squeeze(1)
        fused = torch.cat([t, a], dim=-1)
        return {
            'fused_logits': self.fuse(fused),
            'text_logits':  self.head_text(t),
            'audio_logits': self.head_audio(a),
        }

def jsd(p, q, eps=1e-8):
    """Jensen-Shannon divergence between two probability distributions, base 2.
    p, q: (B, C). Returns (B,) in [0, 1]."""
    m = 0.5 * (p + q)
    kl_pm = (p * (p.clamp_min(eps).log2() - m.clamp_min(eps).log2())).sum(-1)
    kl_qm = (q * (q.clamp_min(eps).log2() - m.clamp_min(eps).log2())).sum(-1)
    return 0.5 * (kl_pm + kl_qm)

fusion = CMAFFusion().to(device)
print('Fusion params (M):', sum(p.numel() for p in fusion.parameters() if p.requires_grad) / 1e6)

## 8 · Train (10 epochs, fast — only fusion params)

In [ ]:
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
import time

EPOCHS = 10
LR     = 3e-4
optim_ = AdamW(fusion.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
ALPHA_AUX = 0.3  # weight for aux per-stream losses

def evaluate(loader):
    fusion.eval()
    preds, gts = [], []
    with torch.no_grad():
        for batch in loader:
            t = batch['text_emb'].to(device); a = batch['audio_emb'].to(device)
            out = fusion(t, a)
            preds.extend(out['fused_logits'].argmax(-1).cpu().tolist())
            gts.extend(batch['labels'].tolist())
    return accuracy_score(gts, preds), f1_score(gts, preds, average='macro')

best_f1 = 0.0
BEST_CKPT = CKPT_DIR / 'best_fusion_model.pt'
for ep in range(1, EPOCHS + 1):
    fusion.train()
    t0 = time.time(); total = 0.0
    for batch in train_loader:
        t = batch['text_emb'].to(device); a = batch['audio_emb'].to(device)
        y = batch['labels'].to(device)
        out = fusion(t, a)
        loss_fused = criterion(out['fused_logits'], y)
        loss_text  = criterion(out['text_logits'],  y)
        loss_audio = criterion(out['audio_logits'], y)
        loss = loss_fused + ALPHA_AUX * (loss_text + loss_audio)
        optim_.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(fusion.parameters(), 1.0)
        optim_.step()
        total += loss.item()
    val_acc, val_f1 = evaluate(val_loader)
    marker = ''
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({'model_state_dict': fusion.state_dict(),
                    'epoch': ep, 'val_acc': val_acc, 'val_f1': val_f1,
                    'cluster_names': CLUSTER_NAMES}, BEST_CKPT)
        marker = ' ✓ Saved'
    print(f'Epoch {ep}: loss {total/len(train_loader):.4f} | val acc {val_acc:.4f} | val F1 {val_f1:.4f} | {time.time()-t0:.0f}s{marker}', flush=True)

# Test eval
ckpt = torch.load(BEST_CKPT, map_location=device, weights_only=False)
fusion.load_state_dict(ckpt['model_state_dict'])
test_acc, test_f1 = evaluate(test_loader)
print(f'\n🏆 FINAL TEST: acc {test_acc:.4f} | F1 {test_f1:.4f}')
print(f'   Best val ep: {ckpt["epoch"]} | val F1: {ckpt["val_f1"]:.4f}')

## 9 · Sanity-check JSD incongruence on synthetic mixed pairs

In [ ]:
# Create CONGRUENT pairs (same cluster) vs INCONGRUENT pairs (different clusters)
# and verify JSD is higher for incongruent
fusion.eval()

def mean_jsd(pairs, n=500):
    sample = random.sample(pairs, min(n, len(pairs)))
    t = torch.stack([text_embs[ti]   for ti,_,_ in sample]).to(device)
    a = torch.stack([audio_embs[ai]  for _,ai,_ in sample]).to(device)
    with torch.no_grad():
        out = fusion(t, a)
        p_text  = F.softmax(out['text_logits'],  -1)
        p_audio = F.softmax(out['audio_logits'], -1)
        return jsd(p_text, p_audio).mean().item()

# Congruent: from test_pairs (same cluster)
jsd_cong = mean_jsd(test_pairs, n=500)

# Incongruent: pair text with audio of DIFFERENT cluster
incong = []
audio_pool_by_c = {c: (audio_clusters == c).nonzero().squeeze(-1).tolist() for c in range(4)}
for ti, _, tc in test_pairs:
    other_c = random.choice([c for c in range(4) if c != tc and audio_pool_by_c[c]])
    ai = random.choice(audio_pool_by_c[other_c])
    incong.append((ti, ai, tc))
jsd_incong = mean_jsd(incong, n=500)

print(f'Mean JSD on CONGRUENT pairs:   {jsd_cong:.4f}')
print(f'Mean JSD on INCONGRUENT pairs: {jsd_incong:.4f}')
print(f'Ratio (incong/cong):           {jsd_incong / max(jsd_cong, 1e-6):.2f}x')
print()
if jsd_incong > jsd_cong * 1.5:
    print('✓ JSD successfully separates congruent vs incongruent pairs')
else:
    print('⚠ JSD signal weak — auxiliary streams need more training or higher alpha')

## 10 · Upload to HF Model repo

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
REPO = 'Ubaida1/mindsync-fusion-model'
api.create_repo(repo_id=REPO, repo_type='model', exist_ok=True)
api.upload_file(
    path_or_fileobj=str(BEST_CKPT),
    path_in_repo='best_fusion_model.pt',
    repo_id=REPO, repo_type='model',
    commit_message=f'Stage 3: CMAF fusion, test acc {test_acc:.4f}, F1 {test_f1:.4f}, JSD cong/incong {jsd_cong:.3f}/{jsd_incong:.3f}',
)
print('Uploaded → https://huggingface.co/' + REPO)

## Done!

Tell Claude **"Stage 3 done"** with the printed `test acc` / `F1` / `JSD` numbers. He'll:
1. Update the HF Space to load all 3 models (text + audio + fusion)
2. Switch the `/predict` endpoint to real multimodal mode
3. Enable JSD-based clinical incongruence alerts
4. Verify on your phone — full MindSync experience